# Notebook 6: Linear Regression and Regularization
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Derive the closed-form solution for simple linear regression
- Understand the cost function and gradient descent
- Implement multiple linear regression with scikit-learn
- Apply Ridge (L2) and Lasso (L1) regularization
- Understand the bias-variance trade-off

## 1. Intuition: Fitting a Line

Given data points $(x_i, y_i)$, we want to find the line:

$$\hat{y} = w_0 + w_1 x$$

that minimises the **Mean Squared Error** (MSE):

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

The parameters $w_0$ (intercept) and $w_1$ (slope) have a **closed-form solution** (Ordinary Least Squares):  
$$w_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}, \quad w_0 = \bar{y} - w_1 \bar{x}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Generate synthetic data: house size → price
size = np.random.uniform(500, 3500, 80)
price = 50 + 0.15 * size + np.random.normal(0, 30, 80)  # $k

# OLS closed form
xm, ym = size.mean(), price.mean()
w1 = np.sum((size - xm) * (price - ym)) / np.sum((size - xm)**2)
w0 = ym - w1 * xm
print(f'OLS: price = {w0:.2f} + {w1:.4f} × size')

x_line = np.linspace(size.min(), size.max(), 100)
y_line = w0 + w1 * x_line

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(size, price, alpha=0.6, label='Data')
ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'y = {w0:.1f} + {w1:.4f}x')
ax.set_xlabel('Size (sq ft)')
ax.set_ylabel('Price ($k)')
ax.set_title('Simple Linear Regression')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Gradient Descent — Learning the Parameters Iteratively

For large datasets, computing the closed-form solution is expensive. **Gradient Descent** updates parameters incrementally:

$$w \leftarrow w - \alpha \frac{\partial \text{MSE}}{\partial w}$$

where $\alpha$ is the **learning rate** — how big each step is.

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def gradient_descent(X, y, lr=1e-7, epochs=2000):
    w0, w1 = 0.0, 0.0
    n = len(y)
    history = []
    for _ in range(epochs):
        y_pred = w0 + w1 * X
        dw0 = -2/n * np.sum(y - y_pred)
        dw1 = -2/n * np.sum((y - y_pred) * X)
        w0 -= lr * dw0
        w1 -= lr * dw1
        history.append(mse(y, y_pred))
    return w0, w1, history

w0_gd, w1_gd, hist = gradient_descent(size, price)
print(f'GD: price = {w0_gd:.2f} + {w1_gd:.4f} × size')
print(f'OLS: price = {w0:.2f} + {w1:.4f} × size')

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(hist)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('Loss Curve During Training')
plt.tight_layout()
plt.show()

## 3. Multiple Linear Regression

With $p$ features:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_p x_p$$

In matrix form:

$$\hat{\mathbf{y}} = X \mathbf{w}$$

This is exactly what `LinearRegression` in scikit-learn solves.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target  # target = median house value

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE : {rmse:.3f}')
print(f'R²   : {r2_score(y_test, y_pred):.3f}')

# Feature coefficients
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print('\nTop feature coefficients:')
print(coef_df.head())

In [ ]:
# Residual analysis
residuals = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(y_pred, residuals, alpha=0.4, s=10)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residual Plot')

axes[1].hist(residuals, bins=40, edgecolor='white', color='steelblue')
axes[1].set_title('Distribution of Residuals')
axes[1].set_xlabel('Residual')
plt.tight_layout()
plt.show()

## 4. The Bias-Variance Trade-off

| | High Bias | High Variance |
|--|-----------|---------------|
| **Symptom** | Underfitting | Overfitting |
| **Train error** | High | Low |
| **Test error** | High | High |
| **Model** | Too simple | Too complex |

**Solution**: Regularization — add a penalty term to the loss function to discourage large weights.

## 5. Ridge (L2) Regularization

$$\text{Loss} = \text{MSE} + \lambda \sum_{j=1}^p w_j^2$$

Large $\lambda$ → stronger regularization → smaller weights → simpler model.

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def eval_model(name, model, X_train, X_test, y_train, y_test):
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f'{name:20s}  RMSE={np.sqrt(mean_squared_error(y_test, pred)):.3f}  R²={r2_score(y_test, pred):.3f}')

eval_model('LinearRegression',   LinearRegression(),       X_train, X_test, y_train, y_test)
eval_model('Ridge (α=0.1)',       Ridge(alpha=0.1),        X_train, X_test, y_train, y_test)
eval_model('Ridge (α=10)',        Ridge(alpha=10),         X_train, X_test, y_train, y_test)
eval_model('Ridge (α=100)',       Ridge(alpha=100),        X_train, X_test, y_train, y_test)
eval_model('Lasso (α=0.01)',      Lasso(alpha=0.01),       X_train, X_test, y_train, y_test)
eval_model('ElasticNet (α=0.1)',  ElasticNet(alpha=0.1),   X_train, X_test, y_train, y_test)

## 6. Lasso (L1) — Built-in Feature Selection

$$\text{Loss} = \text{MSE} + \lambda \sum_{j=1}^p |w_j|$$

Lasso drives some coefficients to **exactly zero**, effectively selecting features. This makes it useful for high-dimensional data.

In [ ]:
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]
coefficients = []

for a in alphas:
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=a, max_iter=5000))])
    pipe.fit(X_train, y_train)
    coefficients.append(pipe.named_steps['model'].coef_)

fig, ax = plt.subplots(figsize=(10, 5))
for i, feat in enumerate(X.columns):
    ax.plot(alphas, [c[i] for c in coefficients], marker='o', label=feat)
ax.set_xscale('log')
ax.axhline(0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('α (regularization strength)')
ax.set_ylabel('Coefficient value')
ax.set_title('Lasso Coefficient Paths')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()
print('As α increases, more coefficients shrink to zero (feature selection)')

## Exercises

1. Implement the **closed-form OLS** solution for multiple linear regression using the matrix formula: `w = (XᵀX)⁻¹ Xᵀy`. Compare coefficients with scikit-learn.
2. Use `RidgeCV` to find the best `alpha` via cross-validation on the California housing dataset.
3. Create a polynomial regression model (degree 3) using `PolynomialFeatures`. Does it overfit? How does Ridge regularization help?
4. Plot the **learning curve** (train vs validation error as training size increases). What does it tell you about bias and variance?

## Reflection Questions
- Why does L1 (Lasso) produce sparse coefficients while L2 (Ridge) does not? (Visualise the constraint regions.)
- If you double all your features, what happens to the OLS solution? What happens to Ridge?
- When would you prefer Lasso over Ridge?

---
**Next →** `07_logistic_regression.ipynb`